<a href="https://colab.research.google.com/github/hiiamlars/analytical_flexibility_llm_reviews/blob/main/data_generation/llm_review_judgments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup

In [5]:
# @title Dependencies

# Installation
!pip install -q -U accelerate peft transformers bitsandbytes
!pip install -q pyarrow
!pip install -q tqdm

# Third-party imports
from google.colab import drive
import google.colab.userdata
import pandas as pd
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from huggingface_hub import login
import os
from tqdm.auto import tqdm

In [2]:
# @title Path definitions

# Mount GDrive
drive.mount('/content/drive')

# Set working directory
WORKING_DIR = "/content/drive/MyDrive/Analytical_flexibility_llm_reviews"

# Create working directory
os.makedirs(WORKING_DIR, exist_ok=True)
print(f"Ensured working directory exists at: {WORKING_DIR}.")

# Define and create DATASET_DIR to load the paper content
DATASET_DIR = os.path.join(WORKING_DIR, "llm_reviews_processed")
os.makedirs(DATASET_DIR, exist_ok=True)
print(f"Ensured dataset directory exists at: {DATASET_DIR}.")

# Define and create the RAW_DIR for the llm reviews (and their logs)
RAW_DIR = os.path.join(WORKING_DIR, "llm_reviews_judged")
os.makedirs(RAW_DIR, exist_ok=True)
print(f"Ensured raw directory exists at: {RAW_DIR}.")

# Define the full result path (final Parquet output)
RESULTS_PATH = os.path.join(RAW_DIR, "llm_reviews_evaluated.parquet")
# Define the full result path for intermediate JSONL storage
RESULTS_JSON_PATH = os.path.join(RAW_DIR, "llm_reviews_evaluated.jsonl")

# Define the full log path (final Parquet output)
LOG_PATH = os.path.join(RAW_DIR, "llm_reviews_log.parquet")
# Define the full log path for intermediate JSONL storage
LOG_JSON_PATH = os.path.join(RAW_DIR, "llm_reviews_log.jsonl")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ensured working directory exists at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews.
Ensured dataset directory exists at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews_processed.
Ensured raw directory exists at: /content/drive/MyDrive/Analytical_flexibility_llm_reviews/llm_reviews_judged.


In [7]:
# @title UserData HuggingFace

# Get HuggingFace token key
hf_token = userdata.get('HUGGINGFACE_TOKEN')

# Login to HuggingFace via token key
login(token=hf_token)

NameError: name 'userdata' is not defined

In [ ]:
# @title Load LLM-judge

# 0. Specify Base model and Adapter
model_id = "meta-llama/Llama-3.1-8B-Instruct"
adapter_id = "boda/RevUtil_Llama-3.1-8B-Instruct_score_only"

# 1. Setup bit configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

# 2. Load Tokenizer and Base Model
tokenizer = AutoTokenizer.from_pretrained(model_id)
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# 3. Add the Adapter
model = PeftModel.from_pretrained(base_model, adapter_id)
model.eval() # Put model in 'judgment' mode

In [ ]:
TASK_DESCRIPTION_HEADER = '''###Task Description: You are an expert in evaluating peer review comments with respect to different aspects. These aspects are aimed to maximize the utilization of the review comments for the authors. The primary purpose of the review is to help/guide authors in improving their drafts. Keep this in mind while evaluating the review point. Whenever you encounter a borderline case, think: “Will this review point help authors improve their draft?”. There is no correlation between the aspect score and the length of the review point.'''

ASPECTS_NO_EXAMPLES = {
"actionability":
'''
**Actionability**

**Definition:** Measures the level of actionability in a review point. We evaluate actionability based on two criteria:

1. **Explicit vs. Implicit**:
   - **Explicit:** Actions or suggestions that are direct or apparent. Authors can directly identify modifications they should apply to their draft. Clarification questions should be treated as explicit statements if they give a direct action.
   - **Implicit:** Actions that need to be inferred from the comment. This includes missing parts that need to be added. Authors can deduce what needs to be done after reading the comment.

2. **Concrete vs. Vague**:
   - **Concrete:** Once the action is identified, the authors know exactly what needs to be done and how to apply the action.
   - **Vague:** After identifying the action, the authors still don’t know how to carry out this action.

**Importance:** It’s more important for actions to be concrete so that authors know how to apply them. It's also preferred for actions to be stated directly rather than inferred.

**Actionability Scale (1-5):**

1. **1: Unactionable**
   - **Definition:** The comment lacks meaningful information to help authors improve the paper. Authors do not know what they should do after reading the comment.

2. **2: Borderline Actionable**
   - **Definition:** The comment includes an implicitly stated action or an action that can be inferred. However, the action itself is vague and lacks detail on how to apply it.

3. **3: Somewhat Actionable**
   - **Definition:** The comment explicitly states an action but is vague on how to execute it.

4. **4: Mostly Actionable**
   - **Definition:** The comment implicitly states an action but concretely states how to implement the inferred action.

5. **5: Highly Actionable**
   - **Definition:** The comment contains an explicit action and concrete details on how to implement it. Authors know exactly how to apply it.
''',

"grounding_specificity": '''

**Grounding Specificity**

**Definition:** Measures how explicitly a review comment refers to a specific part of the paper and how clearly it identifies the issue with that part. This helps authors understand what needs revision and why. Grounding specificity has two key components:

1. **Grounding:** How well the authors can identify the specific part of the paper being addressed.
   - **Weak Grounding:** The author can make an educated guess but cannot precisely identify the referenced part.
   - **Full Grounding:** The author can accurately pinpoint the section, table, figure, or unique aspect being addressed. This can be achieved through:
     - Literal mentions of sections, tables, figures, etc.
     - Mentions of unique elements of the paper.
     - General comments that clearly imply the relevant parts without explicitly naming them.

2. **Specificity:** How clearly the comment details what is wrong or missing in the referenced part. If external work is mentioned, it also evaluates whether specific examples are provided.

**Importance:** It's more important for the comment to be grounded than to be specific.

**Grounding Specificity Scale (1-5):**

1. **Not Grounded**
   - **Definition**: This comment is not grounded at all. It does not identify a specific area in the paper. The comment is highly unspecific.

2. **Weakly Grounded and Not Specific**
   - **Definition**: The authors cannot confidently determine which part the comment addresses. Further, the comment does not specify what needs to be addressed in this part.

3. **Weakly Grounded and Specific**
   - **Definition**: The authors cannot confidently determine which part the comment addresses. However, the comment clearly specifies what needs to be addressed in this part.

4. **Fully Grounded and Under-Specific**
   - **Definition**: The comment explicitly mentions which part of the paper it addresses, or it should be obvious to the authors. However, this comment does not specify what needs to be addressed in this part.

5. **Fully Grounded and Specific**
   - **Definition**: The comment explicitly mentions which part of the paper it addresses, and it is obvious to the authors. The comment specifies what needs to be addressed in this part.
''',

"verifiability":
'''
**Verifiability**

**Definition:** Evaluates whether a review comment contains a claim and, if so, how well that claim is supported using logical reasoning, common knowledge, or external references.

### **Step 1: Claim Extraction**

**Objective:**
Determine whether the given text contains a claim (i.e., an opinion, judgment, or suggestion) or consists solely of factual statements that require no verification.

**Claim Definition:**
A statement is considered a claim if it falls into one or more of the following categories:
- **Subjective opinions or disagreements** (e.g., criticism of an experimental choice).
- **Suggestions or requests for changes** (e.g., recommending removal, addition, or discussion).
- **Judgments about the paper** (e.g., stating something is unclear, not well-written, or lacks detail).
- **Deductions or inferred observations** that go beyond merely stating facts.
- **Statements requiring justification** to be understood or accepted.


**Normal Statements ("No Claim")**
A statement is classified as "X" if it:
- Describes facts without suggesting changes.
- Makes general statements about the paper without an opinion.
- Presents objective, verifiable facts that require no justification.
- Asks for clarifications or general questions.
- States logical statements or directly inferable information.
- Makes positive claims (e.g., *“The paper is well-written”*), as these do not help improve the work.

---

### **Step 2: Verifiability Verification**

**Objective:**
Assess how well a claim is verified by examining the reasoning, common knowledge, or external references provided. The purpose is to ensure that the review comment helps the authors improve their work.

**Verification Methods:**
A claim is considered verifiable if supported by one or more of the following:
- **Logical reasoning** – A clear explanation of why the claim is valid.
- **Common knowledge** – Reference to well-accepted practices or standards.
- **External references** – Citation of relevant literature, data, or sources.

**Verifiability Scale (1–5 & X):**

1. **1: Unverifiable**
   - The comment contains a claim without any supporting evidence or justification.
2. **2: Borderline Verifiable**
   - Some support is provided, but it is vague, insufficient, or difficult to follow.
3. **3: Somewhat Verifiable**
   - The claim has some justification but lacks key elements (e.g., examples, references).
4. **4: Mostly Verifiable**
   - The claim is well-supported but has minor gaps in explanation or references.
5. **5: Fully Verifiable**
   - The claim is thoroughly supported by explicit, sufficient, and robust evidence, such as:
     - Clear reasoning and precise explanations.
     - Specific references to external works.
     - Logical and unassailable common-sense arguments.
6. **X: No Claim**
- The comment contains only factual, descriptive statements without claims, opinions, or suggestions.

---

### **Instructions for Evaluation:**
1. **Extract Claims:** Determine whether the review comment contains a claim or is a normal statement. If there is no claim, score it as "X"
2. **Assess Verifiability:** If a claim exists, score it based on how well it is justified from 1 to 5.
'''
,

"helpfulness" : '''
**Helpfulness**

**Definition:** Assign a subjective score to reflect the value of the review comment to the authors. Helpfulness is rated on a scale from 1 to 5, with the following definitions:

1. **1: Not Helpful at All**
   - **Definition:** The comment fails to identify meaningful weaknesses or suggest improvements, leaving the authors with no actionable feedback.

2. **2: Barely Helpful**
   - **Definition:** The comment identifies a weakness or improvement area but is vague, lacks clarity, or provides minimal guidance, making it only slightly beneficial for the authors.

3. **3: Somewhat Helpful**
   - **Definition:** The comment identifies weaknesses or areas for improvement but is incomplete or lacks depth. While the authors gain some insights, the feedback does not fully address their needs for improving the draft.

4. **4: Mostly Helpful**
   - **Definition:** The comment provides clear and actionable feedback on weaknesses and areas for improvement, though it could be expanded or refined to be fully comprehensive and impactful.

5. **5: Highly Helpful**
   - **Definition:** The comment thoroughly identifies weaknesses and offers detailed, actionable, and constructive suggestions that empower the authors to significantly improve their draft.
'''
}

INSTRUCTION_SCORE_ONLY_PROMPT_TAIL = '''
###Instruction:
Evaluate the review based on the given definitions of the aspect(s) above. Output only the score. The possbile values for scores are 1-5 and X.

Generate the output in JSON format with the following format:

   "actionability_label": "",
   "grounding_specificity_label": "",
   "verifiability_label": "",
   "helpfulness_label": ""


###Review Point:
{review_point}''' ### According to the repo code, but with deviation from the examplary prompt in the paper

In [ ]:
# @title Data import

# Read processed llm reviews
df_reviews = pd.read_parquet(os.path.join(DATASET_DIR, "llm_reviews_segmented.parquet"))

# Check df_reviews
display(df_reviews.head(3))
display(df_reviews.shape)

In [ ]:
# @title Subsetting (if active)

# Define the number of rows to subset. Set to None to disable subsetting.
num_rows_to_subset = 3  # Example: subset to the first 10 rows

if num_rows_to_subset is not None and num_rows_to_subset > 0:
    df_reviews = df_reviews.head(num_rows_to_subset)
    print(f"DataFrame subsetted to the first {num_rows_to_subset} rows.")

# Check subsetted df_reviews
display(df_reviews.head())
display(df_reviews.shape)

## Function

In [ ]:
# @title Scoring logic

def get_scores(comment):

    prompt_parts = []
    prompt_parts.append(TASK_DESCRIPTION_HEADER)

    # Iterate through aspects to add their definitions dynamically
    for aspect_key, aspect_definition_text in ASPECTS_NO_EXAMPLES.items():
        # Adjust aspect name for display as per the desired format
        if aspect_key == "grounding_specificity":
            display_aspect_name = "Grounding & Specificity"
        else:
            display_aspect_name = aspect_key.replace('_', ' ').title() # Capitalize and replace underscore

        # Combine the aspect name and its definition
        prompt_parts.append(f"Aspect: {display_aspect_name} {aspect_definition_text}")

    # Add the instruction and review point tail, formatted with the actual comment
    prompt_parts.append(INSTRUCTION_SCORE_ONLY_PROMPT_TAIL.format(review_point=comment))

    # Join all parts to form the final prompt string
    prompt = "\n".join(prompt_parts)

    # Tokenize review text
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    # Model call
    with torch.no_grad():
        outputs = model.generate(**inputs)

    # Decode only the newly generated tokens (excluding the input prompt)
    # return tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return prompt


## Execution

In [ ]:
# @title Scoring

# Initialize an empty list to store the predicted scores
predicted_scores = []

# Iterate over comments
for comment in tqdm(df_reviews['segmented_comment'], desc="Scoring review comments"):
    # Call the get_scores function
    score_output = get_scores(comment)
    # Append the scores to a list
    predicted_scores.append(score_output)

# Add score list as a new column
df_reviews['predicted_score'] = predicted_scores

# Display df_reviews
display(df_reviews.head())

In [ ]:
# Print each predicted score as simple text
for score in df_reviews['predicted_score']:
    print(score)
    print("\n---\n") # Add a separator for readability

## Results

In [ ]:
# ToDo: Convert the results from .json to .parquet

In [ ]:
# ToDo: Display head and dimensions of the result